# 3 — Computing the inverse solution

## 3.1 The forward model

An EEG electrode does not see one brain region. Every sensor sees a weighted sum of the activity of the *whole* cortex, because current spreads through brain, skull and scalp. In the quasi-static approximation of Maxwell's equations that mixing is **linear**, which is the single fact that makes everything below possible:

$$\mathbf{y}(t) \;=\; \mathbf{L}\,\mathbf{x}(t) \;+\; \mathbf{n}(t)$$

| symbol | shape | meaning |
|---|---|---|
| $\mathbf{y}(t)$ | $N \times 1$ | measured sensor data at time $t$ — $N$ channels |
| $\mathbf{L}$ | $N \times M$ | **lead field** (gain matrix) — $M$ candidate sources |
| $\mathbf{x}(t)$ | $M \times 1$ | source amplitudes we want to recover |
| $\mathbf{n}(t)$ | $N \times 1$ | additive noise |

Stacking $T$ time samples gives $\mathbf{Y} = \mathbf{L}\mathbf{X} + \mathbf{N}$, with $\mathbf{Y}$ of shape $N \times T$ and $\mathbf{X}$ of shape $M \times T$.

**Read $\mathbf{L}$ column by column.** Column $k$ is the scalp topography produced by a unit-strength dipole at source point $k$. So the lead field is a dictionary of topographies, one per candidate source, and the forward model just says: *the measurement is a weighted sum of these topographies.*

Crucially, $\mathbf{L}$ depends only on **geometry and conductivity** — the source space, the BEM head model and the electrode positions — never on the data. It is computed once from the head model and the source space, and inside MNE's forward-solution object it is a plain array you can inspect:

`fwd['sol']['data']` → shape `(n_channels, n_sources)`

With *free* orientations each source point carries three orthogonal dipoles, so $\mathbf{L}$ becomes $N \times 3M$ and $\mathbf{x}$ stacks the $(x, y, z)$ components per location. With *fixed* orientations (dipoles clamped normal to the cortical surface) it is $N \times M$ again.

> MNE's documentation writes the same equation as $b = G j$. Different letters, identical model.

## 3.2 The inverse problem is ill-posed

We know $\mathbf{y}$, we know $\mathbf{L}$, and we want $\mathbf{x}$. We cannot simply invert $\mathbf{L}$, for two independent reasons.

**1. It is underdetermined.** A typical source space has $M \approx 5\,000$–$20\,000$ sources, while $N$ is at most a few hundred channels, so $M \gg N$. The lead field therefore has a large null space: there exist non-zero source patterns $\mathbf{x}_0$ with

$$\mathbf{L}\,\mathbf{x}_{0} = \mathbf{0}$$

These are **silent sources** — activity that produces literally no measurement. Any multiple of $\mathbf{x}_0$ can be added to a solution without changing the data by one microvolt, so infinitely many source configurations explain the same recording *exactly*. No amount of clean data fixes this; it is a property of the physics, not of the noise.

**2. It is ill-conditioned.** Even inside the row space of $\mathbf{L}$, deep and radially oriented sources project onto the sensors very weakly. Those directions correspond to tiny singular values of $\mathbf{L}$, and naively inverting them multiplies the measurement noise by enormous factors.

The consequence: **the inverse problem has no unique solution, so we must add assumptions.** Every method below is a different assumption about which of the infinitely many candidate solutions we prefer. Choosing an inverse method *is* choosing a prior — it is not a neutral technical step.

## 3.3 All the methods here are linear filters

Every method in this notebook estimates the sources with a single matrix $\mathbf{W}$ of shape $M \times N$, applied to the data:

$$\hat{\mathbf{x}}(t) \;=\; \mathbf{W}\,\mathbf{y}(t)$$

$\mathbf{W}$ is the *inverse operator*, and row $k$ of $\mathbf{W}$ is the **spatial filter** for source $k$. The methods differ only in how $\mathbf{W}$ is constructed — which is why MNE separates building the operator from applying it (`make_inverse_operator` / `apply_inverse`, `make_lcmv` / `apply_lcmv`).

### Minimum-norm estimates (MNE, dSPM, sLORETA)

Pick the solution that fits the data while keeping total source power small:

$$\hat{\mathbf{x}} \;=\; \arg\min_{\mathbf{x}}\;
\underbrace{\left\lVert \mathbf{y}-\mathbf{L}\mathbf{x} \right\rVert^{2}_{\mathbf{C}^{-1}}}_{\text{data fit}}
\;+\; \lambda^{2}\,\underbrace{\left\lVert \mathbf{x} \right\rVert^{2}_{\mathbf{R}^{-1}}}_{\text{prior}}$$

which has the closed-form solution

$$\mathbf{W} \;=\; \mathbf{R}\mathbf{L}^{\mathsf{T}}
\left( \mathbf{L}\mathbf{R}\mathbf{L}^{\mathsf{T}} + \lambda^{2}\mathbf{C} \right)^{-1}$$

| symbol | what it is | where it comes from in MNE |
|---|---|---|
| $\mathbf{C}$ | noise covariance — whitens the sensors | `noise_cov`, estimated from the baseline |
| $\mathbf{R}$ | source covariance — the prior | shaped by `depth` and `loose` |
| $\lambda^{2}$ | regularisation strength | `lambda2 = 1.0 / snr ** 2` |

$\mathbf{R} = \mathbf{I}$ is plain MNE: *the smallest total source power that explains the data*. This prior is biased towards superficial sources, which is what `depth=0.8` compensates for. **dSPM** and **sLORETA** keep the same $\mathbf{W}$ but rescale each row — by the projected noise (dSPM) or by the estimated source variance (sLORETA) — turning amplitudes into noise-normalised statistical maps.

### LCMV beamformer

A beamformer builds each spatial filter separately, and uses the **data** covariance $\mathbf{C}_d$ rather than a fixed prior. For source $k$ with lead field $\mathbf{L}_k$:

$$\mathbf{w}_{k} \;=\; \arg\min_{\mathbf{w}}\; \mathbf{w}^{\mathsf{T}}\mathbf{C}_{d}\mathbf{w}
\qquad \text{subject to} \qquad \mathbf{w}^{\mathsf{T}}\mathbf{L}_{k} = 1$$

*Minimise total output power, but keep unit gain at the source of interest.* Anything the filter passes that is not source $k$ costs variance, so the filter learns to suppress it. The solution is

$$\mathbf{w}_{k} \;=\;
\frac{\mathbf{C}_{d}^{-1}\mathbf{L}_{k}}{\mathbf{L}_{k}^{\mathsf{T}}\mathbf{C}_{d}^{-1}\mathbf{L}_{k}}$$

Because $\mathbf{C}_d$ is estimated from the data and must be inverted, it is regularised as $\mathbf{C}_d + \alpha \,\frac{\operatorname{tr}(\mathbf{C}_d)}{N}\mathbf{I}$ — that is the `reg=0.05` argument. With `weight_norm="unit-noise-gain"` the filter is instead normalised by $\lVert \mathbf{w}_k \rVert$, so that projected noise is constant across the brain and deep sources stop looking artificially quiet.

Two consequences worth carrying into the exercises: a beamformer is **adaptive** (change the time window or the filter band and every spatial filter changes), and it assumes sources are not perfectly correlated — two sources locked in phase partly cancel each other.

## 3.4 What you actually get back

Substituting the forward model into the estimate shows what any linear method really returns:

$$\hat{\mathbf{x}} \;=\; \mathbf{W}\mathbf{y}
\;=\; \underbrace{\mathbf{W}\mathbf{L}}_{\textstyle \mathbf{Res}}\,\mathbf{x} \;+\; \mathbf{W}\mathbf{n}$$

$\mathbf{Res} = \mathbf{W}\mathbf{L}$ is the **resolution matrix**. Perfect reconstruction would mean $\mathbf{Res} = \mathbf{I}$; it never is, for any method, because of the null space in §3.2.

- **Column $k$** — the *point-spread function*: how a single source at $k$ smears across your map.
- **Row $k$** — the *cross-talk function*: how much activity from everywhere else leaks into the time course you extract at $k$.

This is source leakage, and it is the reason a parcel time course is never "the activity of that parcel". Keep $\mathbf{Res}$ in mind before interpreting any connectivity or ROI result.

## 3.5 In this notebook

1. Load the epochs and the noise and data covariances computed in notebook 02.
2. Obtain the forward solution for this head model and source space — the lead field $\mathbf{L}$ of §3.1.
3. Build $\mathbf{W}$ with `make_inverse_operator` and apply dSPM.
4. Build $\mathbf{W}$ with `make_lcmv` and apply the beamformer.
5. Compare the two on the same data — and look at what each one does to a known source.